In [2]:
import sys
import os
from typing import List, Dict, Any
import importlib
import pandas as pd
from datetime import datetime, timedelta
from pandas.tseries.offsets import BDay

In [3]:
# Setup dell'ambiente
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

def reload_modules() -> None:
    """Ricarica i moduli necessari."""
    modules_to_reload = [
        'lib.data_processing', 'lib.visualization',
        'lib.signal_combination', 'lib.strategy', 'lib.utils',
        'lib.dash.integrated_dashboard',
        'lib.signals.indicators', 'lib.signals.signals_BB', 'lib.signals.signals_CCI', 'lib.signals.signals_EMA', 'lib.signals.signals_MACD', 'lib.signals.signals_RSI', 'lib.signals.signals_SMA'
    ]
    for module in modules_to_reload:
        if module in sys.modules:
            importlib.reload(sys.modules[module])

In [4]:
# Importa i moduli necessari
from lib.data_processing import *
from lib.visualization import *
from lib.utils import TradingStrategyInput, get_user_input, export_priceaction_to_excel
from lib.strategy import *
from lib.signal_combination import *

from lib.signals.indicators import *
from lib.signals.signals_BB import *
from lib.signals.signals_CCI import *
from lib.signals.signals_EMA import *
from lib.signals.signals_MACD import *
from lib.signals.signals_RSI import *
from lib.signals.signals_SMA import *

from lib.dash.integrated_dashboard import run_dashboard

d:\03_Coding\Python\SearchForAlpha_lab\lib\dash\integrated_dashboard.py:4: UserWarning: 
The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`
  import dash_html_components as html
d:\03_Coding\Python\SearchForAlpha_lab\lib\dash\integrated_dashboard.py:5: UserWarning: 
The dash_table package is deprecated. Please replace
`import dash_table` with `from dash import dash_table`

Also, if you're using any of the table format helpers (e.g. Group), replace 
`from dash_table.Format import Group` with 
`from dash.dash_table.Format import Group`
  from dash_table import DataTable


In [5]:
today = datetime.now().date()
# Se oggi non è un giorno lavorativo, trova l'ultimo giorno lavorativo
if today.weekday() >= 5:  # 5 = Sabato, 6 = Domenica
    DEFAULT_END_DATE = (today - BDay(1)).strftime('%Y-%m-%d')
else:
    DEFAULT_END_DATE = today.strftime('%Y-%m-%d')

In [12]:
def main(default_ticker, default_start_date, default_end_date,
         default_initial_capital, round_down):
    
    # Fetch initial data and add indicators
    df = fetch_data(default_ticker, default_start_date, default_end_date)
    df = add_indicators(df)
    
    # Generate signals
    df, signal_headers = generate_signals(df)
    
    signal_combos_results, best_buy, best_sell, best_value, output_file = test_all_combinations(
    df=df,
    initial_capital=default_initial_capital,
    combination_type='Buy_&_Sell',
    max_combinations=5000,
    max_signals=3
        )
    
    print(f"\nTest run completed. Results saved to {output_file}")

    exported_file1 = export_priceaction_to_excel(default_ticker, signal_combos_results, export_type='combination')
    os.startfile(exported_file1)

    return signal_combos_results, best_buy, best_sell, best_value, output_file
    
if __name__ == "__main__":
    main(
    default_ticker='AMZN', 
    default_start_date="2020-06-01", 
    default_end_date=DEFAULT_END_DATE,          
    default_initial_capital=10000, 
    
    
    round_down=True
    
    )   

Available buy signals: ['BB_Breakout_Buy', 'BB_MeanReversion_Buy', 'BB_Squeeze_Buy', 'BB_DoubleBottom_Buy', 'MACD_ZeroCross_Buy', 'MACD_SignalCross_Buy', 'MACD_Histogram_Buy', 'RSI_Oversold_Buy', 'CCI_Oversold_Buy', 'CCI_Reversal_Buy', 'CCI_ZeroCross_Buy', 'SMA_TripleCross_Buy', 'SMA_PriceCross_Buy', 'SMA_TrendFollow_Buy', 'EMA_TripleCross_Buy', 'EMA_Distance_Buy', 'EMA_Momentum_Buy', 'EMA_ValueZone_Buy', 'EMA_Divergence_Buy', 'EMA_Volatility_Buy']
Available sell signals: ['BB_Breakout_Sell', 'BB_MeanReversion_Sell', 'BB_Squeeze_Sell', 'BB_DoubleTop_Sell', 'MACD_ZeroCross_Sell', 'MACD_SignalCross_Sell', 'MACD_Histogram_Sell', 'RSI_Overbought_Sell', 'CCI_Overbought_Sell', 'CCI_Reversal_Sell', 'CCI_ZeroCross_Sell', 'SMA_TripleCross_Sell', 'SMA_PriceCross_Sell', 'SMA_TrendFollow_Sell', 'EMA_TripleCross_Sell', 'EMA_Distance_Sell', 'EMA_Momentum_Sell', 'EMA_ValueZone_Sell', 'EMA_Divergence_Sell', 'EMA_Volatility_Sell']
Maximum signals per combination: 3
Results will be saved to: d:\03_Codin

Processing chunks:   0%|          | 0/50 [00:00<?, ?it/s]

Results saved to d:\03_Coding\Python\SearchForAlpha_lab\results\signal_combination_results.parquet
Columns in the results DataFrame: ['Buy_Signals', 'Sell_Signals', 'Buy_Signals_Count', 'Sell_Signals_Count', 'Final_Portfolio_Value', 'Total_Return', 'Annual_Return', 'Max_Drawdown', 'Sharpe_Ratio', 'Win_Rate', 'Profit_Factor', 'Average_Trade_Duration']
First 5 results:
            Buy_Signals                Sell_Signals  Buy_Signals_Count  \
0  ('BB_Breakout_Buy',)       ('BB_Breakout_Sell',)                 78   
1  ('BB_Breakout_Buy',)  ('BB_MeanReversion_Sell',)                 78   
2  ('BB_Breakout_Buy',)        ('BB_Squeeze_Sell',)                 78   
3  ('BB_Breakout_Buy',)      ('BB_DoubleTop_Sell',)                 78   
4  ('BB_Breakout_Buy',)    ('MACD_ZeroCross_Sell',)                 78   

   Sell_Signals_Count  Final_Portfolio_Value  Total_Return  Annual_Return  \
0                  50                   8225       -0.1775      -0.044092   
1                  60          